# RetailMind AI — Day 2: Decision Tree, Random Forest & SVM

**Goal:** Upgrade the churn-classification baseline by training three models and comparing their performance, with special attention to **recall** (finding customers who are likely to churn).

**Important:** This is a new Day 2 notebook. It does not edit `train_churn.ipynb`, `predict_churn.ipynb`, `churn.py`, or the existing `churn_pipeline.joblib`.

### Project assumptions
- Dataset: `data/Telco-Customer-Churn.csv`
- This notebook is saved inside the `notebooks/` folder.
- New Day 2 model files and metrics will be saved separately inside `models/`.


## 1. Import libraries

This increment uses only **scikit-learn**, as specified in the lab instructions, plus pandas/numpy for working with the dataset and joblib for saving models.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load the Telco Customer Churn dataset

The path check supports running the notebook either from the `notebooks/` folder or from the project root.


In [ ]:
DATA_CANDIDATES = [
    Path("../data/Telco-Customer-Churn.csv"),
    Path("data/Telco-Customer-Churn.csv")
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Keep Telco-Customer-Churn.csv inside the project's data/ folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Basic data inspection and cleaning

`TotalCharges` may contain blank strings. Convert it to numeric and let the preprocessing pipeline fill any missing values. Remove `customerID` because it is an identifier, not a useful customer-behaviour feature.


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nMissing values before cleaning:")
print(df.isnull().sum())

# Convert TotalCharges to numeric; invalid/blank values become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove customer identifier if it exists.
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

# Convert target labels to 0 and 1.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Remove rows where the target is missing or could not be converted.
df = df.dropna(subset=["Churn"])
df["Churn"] = df["Churn"].astype(int)

print("\\nCleaned dataset shape:", df.shape)
print("\\nTarget distribution:")
print(df["Churn"].value_counts())


## 4. Separate features and target

- `X` contains the input columns.
- `y` contains the churn label (`1 = churn`, `0 = did not churn`).
- A stratified split keeps the churn/non-churn proportion similar in train and test data.


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("\\nTraining churn proportion:")
print(y_train.value_counts(normalize=True))


## 5. Prepare preprocessing pipelines

Categorical columns are converted into numeric columns using one-hot encoding. Numeric columns are imputed if necessary.

SVM is sensitive to feature scale, so its numeric columns are standardized. Decision Tree and Random Forest do not require scaling, so their numeric values are passed through after imputation.


In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

svm_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Create the three models

We use `class_weight="balanced"` to give more importance to the minority churn class. This can help recall, but the final decision should be based on test metrics rather than assuming one model is best.


In [ ]:
models = {
    "Decision Tree": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=6,
            min_samples_split=10,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "SVM": Pipeline(steps=[
        ("preprocessor", svm_preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])
}

print("Models created:", list(models.keys()))


## 7. Train the models

Each pipeline learns preprocessing only from the training data, then fits its classifier. This avoids fitting encoders/scalers on the test set.


In [ ]:
for model_name, model_pipeline in models.items():
    model_pipeline.fit(X_train, y_train)
    print(model_name, "training completed")


## 8. Evaluate model performance

Metrics:
- **Accuracy:** overall fraction predicted correctly.
- **Precision:** among customers predicted to churn, how many actually churned.
- **Recall:** among customers who actually churned, how many the model detected.
- **F1-score:** balance between precision and recall.
- **ROC-AUC:** ability to rank churners above non-churners.

For a retention team, recall is important because missing a customer who is about to churn can mean losing the chance to contact them.


In [ ]:
results = []
predictions = {}

for model_name, model_pipeline in models.items():
    y_pred = model_pipeline.predict(X_test)
    y_prob = model_pipeline.predict_proba(X_test)[:, 1]

    predictions[model_name] = y_pred

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    by=["Recall", "F1 Score"],
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))


## 9. Compare confusion matrices and classification reports

The confusion matrix layout is:

`[[True Negative, False Positive], [False Negative, True Positive]]`

A **false negative** is a customer who churned but was predicted not to churn. Review this value carefully when comparing churn models.


In [ ]:
for model_name, model_pipeline in models.items():
    y_pred = predictions[model_name]

    print("\\n" + "=" * 60)
    print(model_name)
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))


## 10. Select the best model by recall

For this lab, select the model with the highest test recall. If two models have the same recall, use F1-score as the tie-breaker.

This is a simple lab selection rule, not a universal business rule. In a real deployment, the team should also consider false-positive cost, customer-contact capacity, and a validation set.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)
print("Selected model metrics:")
display(results_df[results_df["Model"] == best_model_name].round(4))


## 11. Save Day 2 models and evaluation results

All outputs use new filenames. The existing Day 1 model file is not overwritten.


In [ ]:
MODELS_DIR_CANDIDATES = [Path("../models"), Path("models")]
MODELS_DIR = next(
    (path for path in MODELS_DIR_CANDIDATES if path.exists()),
    Path("../models")
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save each Day 2 model separately.
for model_name, model_pipeline in models.items():
    safe_name = model_name.lower().replace(" ", "_")
    output_path = MODELS_DIR / f"day2_{safe_name}.joblib"
    joblib.dump(model_pipeline, output_path)
    print("Saved:", output_path)

# Save the selected model with a separate Day 2 filename.
best_model_path = MODELS_DIR / "churn_day2_best_model.joblib"
joblib.dump(best_model, best_model_path)

# Save the comparison table.
metrics_path = MODELS_DIR / "day2_model_metrics.csv"
results_df.to_csv(metrics_path, index=False)

print("\\nBest model saved to:", best_model_path)
print("Metrics saved to:", metrics_path)


## 12. Final Day 2 summary

Complete this section after running all cells. Use the actual values from `results_df`; do not assume a particular algorithm will win.

- Best model by recall: shown in `best_model_name`.
- Compare its precision and F1-score with the other models.
- Check the false-negative count in its confusion matrix.
- Day 2 models and metrics are saved with `day2_` filenames.

### Day 2 completion checklist
- [ ] Dataset loaded and cleaned
- [ ] Train/test split created with stratification
- [ ] Decision Tree trained
- [ ] Random Forest trained
- [ ] SVM trained with scaled numeric features
- [ ] Accuracy, precision, recall, F1 and ROC-AUC compared
- [ ] Confusion matrices and classification reports reviewed
- [ ] Day 2 models and metrics saved separately
